In [ ]:
# import importlib
# from src.utils import statistics
# importlib.reload(statistics)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import numpy as np
import geopandas as gpd
import sys

PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.utils.style import set_plot_style, COLORS, fa
from src.utils.plot_utils import save_figure
from src.utils.statistics import (
    normalize_year,
    normalize_integer,
    missing_by_group,
    hierarchical_impute,
)

set_plot_style()

In [ ]:
DATA_PATH = "../data/raw/divar.csv"
df = pd.read_csv(DATA_PATH, low_memory=False)

In [ ]:
df = df.drop(columns=["Unnamed: 0"])

### Fix DataType

In [ ]:
# rooms_count
df_rooms_count = df.copy()
df_rooms_count["rooms_count"] = pd.Categorical(
    df_rooms_count["rooms_count"],
    categories=["بدون اتاق", "یک", "دو", "سه", "چهار", "پنج یا بیشتر"],
    ordered=True,
)
mapping = {"بدون اتاق": 0, "یک": 1, "دو": 2, "سه": 3, "چهار": 4, "پنج یا بیشتر": 5}

df_rooms_count["rooms_count"] = (
    df_rooms_count["rooms_count"].map(mapping).astype("Int8")
)
df = df_rooms_count

In [ ]:
# construction_year
df_construction_year = df.copy()
df_construction_year["construction_year"] = (
    df_construction_year["construction_year"].apply(normalize_year).astype("Int16")
)
df = df_construction_year

In [ ]:
# total_floors_count
df_total_floors_count = df.copy()
df_total_floors_count["total_floors_count"] = (
    df_total_floors_count["total_floors_count"]
    .apply(lambda x: normalize_integer(x, {"30+": 31, "unselect": pd.NA}))
    .astype("Int8")
)
df = df_total_floors_count

In [ ]:
# unit_per_floor
df_unit_per_floor = df.copy()
df_unit_per_floor["unit_per_floor"] = (
    df_unit_per_floor["unit_per_floor"]
    .apply(lambda x: normalize_integer(x, {"more_than_8": 9, "unselect": pd.NA}))
    .astype("Int8")
)
df = df_unit_per_floor

In [ ]:
# extra_person_capacity
df_extra_person_capacity = df.copy()
df_extra_person_capacity["extra_person_capacity"] = (
    df_extra_person_capacity["extra_person_capacity"]
    .apply(lambda x: normalize_integer(x, {"30+": 30}))
    .astype("Int8")
)
df = df_extra_person_capacity

In [ ]:
# bool_cols
df_bool_cols = df.copy()
bool_cols = [
    "rent_to_single",
    "rent_credit_transform",
    "transformable_price",
    "has_business_deed",
    "has_elevator",
    "has_warehouse",
    "has_parking",
    "is_rebuilt",
    "has_water",
    "has_electricity",
    "has_gas",
    "has_security_guard",
    "has_barbecue",
    "has_pool",
    "has_jacuzzi",
    "has_sauna",
]

df_bool_cols[bool_cols] = df_bool_cols[bool_cols].astype("boolean")
df_bool_cols["has_balcony"] = (
    df_bool_cols["has_balcony"]
    .str.strip()
    .str.lower()
    .replace(
        {
            "true": True,
            "false": False,
            "unselect": pd.NA,
        }
    )
    .astype("boolean")
)
df = df_bool_cols

In [ ]:
float_cols = [
    "rent_value",
    "price_value",
    "credit_value",
    "transformable_credit",
    "transformed_credit",
    "transformable_rent",
    "transformed_rent",
    "land_size",
    "building_size",
    "regular_person_capacity",
    "cost_per_extra_person",
    "rent_price_on_regular_days",
    "rent_price_on_special_days",
    "rent_price_at_weekends",
    "location_latitude",
    "location_longitude",
    "location_radius",
]
df[float_cols] = df[float_cols].astype("float32")

In [ ]:
# floor
df_floor = df.copy()
df_floor["floor"] = (
    df_floor["floor"].apply(lambda x: normalize_integer(x, {"30+": 31})).astype("Int8")
)
df = df_floor

In [ ]:
# created_at_month
df["created_at_month"] = pd.to_datetime(df["created_at_month"])
df["created_year"] = df["created_at_month"].dt.year
df["created_month"] = df["created_at_month"].dt.month
df.drop(columns="created_at_month", inplace=True)

In [ ]:
# category_cols
df_category = df.copy()
category_with_unselect = [
    "deed_type",
    "has_warm_water_provider",
    "has_heating_system",
    "has_cooling_system",
    "has_restroom",
    "building_direction",
    "floor_material",
]
for col in category_with_unselect:
    df_category[col] = df_category[col].replace("unselect", pd.NA)

category_cols = [
    "cat2_slug",
    "cat3_slug",
    "city_slug",
    "user_type",
    "rent_mode",
    "rent_type",
    "price_mode",
    "credit_mode",
    "deed_type",
    "has_warm_water_provider",
    "has_heating_system",
    "has_cooling_system",
    "has_restroom",
    "building_direction",
    "floor_material",
    "property_type",
]
df_category[category_cols] = df_category[category_cols].astype("category")
df = df_category

### Invalid Values

In [ ]:
# location_latitude and location_longitude
iran = gpd.read_file("../data/maps/iran_provinces.geojson")
iran_polygon = iran.union_all()
location_df = df[
    df["location_latitude"].notna() & df["location_longitude"].notna()
].copy()
location_gdf = gpd.GeoDataFrame(
    location_df,
    geometry=gpd.points_from_xy(
        location_df["location_longitude"], location_df["location_latitude"]
    ),
    crs="EPSG:4326",
)
iran_gdf = gpd.GeoDataFrame(geometry=[iran_polygon], crs="EPSG:4326")
inside = gpd.sjoin(location_gdf, iran_gdf, predicate="intersects", how="inner")
inside_index = inside.index
df = pd.concat([df[df["location_latitude"].isna()], df.loc[inside_index]]).sort_index()

In [ ]:
# building_size
df.loc[df["building_size"] > 100_000, "building_size"] = pd.NA

df["building_size"] = df.groupby("cat3_slug")["building_size"].transform(
    lambda s: s.fillna(s.median())
)

In [ ]:
# land_size
df.loc[df["land_size"] > 100_000, "land_size"] = pd.NA

df["land_size"] = df.groupby("cat3_slug")["land_size"].transform(
    lambda s: s.fillna(s.median())
)

In [ ]:
# price_value
df.loc[df["price_value"] == 0, "price_value"] = pd.NA
df.loc[df["price_value"] >= 1e13, "price_value"] = pd.NA

In [ ]:
# rent_value
df.loc[df["rent_value"] >= 1e14, "rent_value"] = pd.NA

In [ ]:
# credit_value
df.loc[df["credit_value"] >= 1e14, "credit_value"] = pd.NA

### Missing Values

In [ ]:
IMPUTE_CONFIG = {
    "building_size": ("median", ["cat3_slug"]),
    "construction_year": ("median", ["cat3_slug"]),
    "rooms_count": ("mode", ["cat3_slug"]),
    "floor": ("mode", ["cat3_slug"]),
    "total_floors_count": ("median", ["cat3_slug"]),
    "unit_per_floor": ("mode", ["cat3_slug"]),
}
for column, (strategy, groups) in IMPUTE_CONFIG.items():
    df[column] = hierarchical_impute(
        df,
        column=column,
        groups=groups,
        strategy=strategy,
    )

### Feature Engineering

In [ ]:
amenity_cols = [
    "has_parking",
    "has_warehouse",
    "has_elevator",
    "has_balcony",
    "has_pool",
    "has_jacuzzi",
    "has_sauna",
    "has_barbecue",
]
df["amenity_count"] = df[amenity_cols].eq(True).sum(axis=1)
df.drop(columns=amenity_cols, inplace=True)

In [ ]:
# df["floor_ratio"] = df["floor"] / df["total_floors_count"]

In [ ]:
# df["units_in_building"] = df["unit_per_floor"] * df["total_floors_count"]

### Selected Features

In [ ]:
rent_df = df[
    (df["cat2_slug"] == "residential-rent")
    | (df["cat2_slug"] == "commercial-rent")
    | (df["cat2_slug"] == "temporary-rent")
].copy()

In [ ]:
(rent_df["credit_value"].isna() & rent_df["rent_value"].notna()).sum()

In [ ]:
rent_df[rent_df["full_deposit"].isna()]["full_deposit"].info()

In [ ]:
df["full_deposit"] = df["credit_value"] + df["rent_value"] * (100_000_000 / 3_000_000)
df.drop(columns=["credit_value", "rent_value"], inplace=True)

In [ ]:
rent_df["full_deposit"].isna().sum()

In [ ]:
rent_df["cat2_slug"].value_counts()

In [ ]:
features = [
    "building_size",
    "city_slug",
    "neighborhood_slug",
    "cat3_slug",
    "rooms_count",
    "building_age",
    "floor_ratio",
    "amenity_count",
    "units_in_building",
    "location_longitude",
]

In [ ]:
rent_df["full_deposit"].info()

In [ ]:
data = rent_df[features + ["full_deposit"]].copy()
data = data.dropna(subset=["full_deposit"])

X = data[features].copy()
y = data["full_deposit"]

### Create Model

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.20,
    random_state=42,
)

In [ ]:
# numeric_features = [
#     "building_size",
#     "rooms_count",
#     "building_age",
#     "floor_ratio",
#     "amenity_count",
#     "location_latitude",
#     "location_longitude",
# ]
# for col in numeric_features:
#     X[col] = pd.to_numeric(X[col], errors="coerce").astype("float32")
# categorical_features = [
#     "city_slug",
#     "neighborhood_slug",
#     "cat3_slug",
#     "has_elevator",
#     "has_parking",
# ]
# for col in categorical_features:
#     X[col] = X[col].astype("string").fillna("Missing").astype(str)

In [ ]:

from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

In [ ]:
model = LinearRegression()

In [ ]:
model.fit(X_train, y_train)

In [ ]:
pred = np.expm1(model.predict(X_test))
y_true = np.expm1(y_test)

print("R2 :", r2_score(y_true, pred))
print("MAE:", mean_absolute_error(y_true, pred))
print("RMSE:", root_mean_squared_error(y_true, pred))